# 05 — AnalyticalRIS final evaluation

Evaluate **analytical_ris** on the exact 1,000 locked test scenarios for each
`N ∈ {16, 32, 64, 96, 128}`. The solver is deterministic; scenario
chunks are used only for wall-clock reliability and are merged back into
one audited raw table.

In [ ]:
from __future__ import annotations

import concurrent.futures
import datetime as dt
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Reproducibility controls
# ---------------------------------------------------------------------------
REPO_URL = "https://github.com/Juliolayme/STAR_RIS_RSMA_TD3.git"
REPO_REF = "agent/td3-qos-scalability-v2"
REPO_COMMIT = "89c39da461523a7f5911a302cb9415aeaa5824ce"
REPO_DIR = Path("/kaggle/working/STAR_RIS_RSMA_TD3")
WORKING_DIR = Path("/kaggle/working")

# Keep Python output unbuffered so Kaggle logs remain useful during long runs.
os.environ["PYTHONUNBUFFERED"] = "1"

def run_command(
    command: Sequence[str],
    *,
    cwd: Path,
    log_path: Path,
    extra_env: dict[str, str] | None = None,
) -> None:
    """Run one subprocess, stream its output, and persist an exact text log.

    The function raises immediately on non-zero exit status. This prevents a
    partially failed experiment from being silently included in the final
    thesis tables.
    """
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)

    print("$", " ".join(map(str, command)))
    with log_path.open("w", encoding="utf-8") as handle:
        process = subprocess.Popen(
            list(map(str, command)),
            cwd=str(cwd),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            handle.write(line)
        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Command failed with exit code {return_code}. See {log_path}"
        )

def clone_and_install() -> str:
    """Clone the declared branch, install it editable, and return the commit SHA."""
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    run_command(
        ["git", "clone", "--filter=blob:none", "--branch", REPO_REF, REPO_URL, REPO_DIR],
        cwd=WORKING_DIR,
        log_path=WORKING_DIR / "git_clone.log",
    )
    run_command(
        ["git", "checkout", "--detach", REPO_COMMIT],
        cwd=REPO_DIR,
        log_path=WORKING_DIR / "git_checkout.log",
    )
    run_command(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-build-isolation",
            "-e",
            str(REPO_DIR),
            "tabulate",
        ],
        cwd=REPO_DIR,
        log_path=WORKING_DIR / "pip_install.log",
    )
    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
    ).strip()
    if commit != REPO_COMMIT:
        raise RuntimeError(f"Repository drift: expected {REPO_COMMIT}, got {commit}")
    print("Repository commit:", commit)
    return commit

def sha256_file(path: Path) -> str:
    """Return the SHA-256 digest of a file without loading it fully into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def create_locked_banks(n_values: Iterable[int], stage_root: Path) -> None:
    """Create the deterministic train/validation/test ScenarioBanks for each N.

    The command and seeds are inherited from the repository's final GitHub
    Actions protocol. Existing complete banks are reused, which makes reruns
    resume-safe after a Kaggle interruption.
    """
    bank_dir = REPO_DIR / "artifacts" / "scenario_banks"
    bank_dir.mkdir(parents=True, exist_ok=True)
    log_dir = stage_root / "logs" / "scenario_banks"

    for n_ris in n_values:
        test_bank = bank_dir / f"N{n_ris}_test.npz"
        if test_bank.exists():
            print(f"Reuse existing ScenarioBank for N={n_ris}: {test_bank}")
            continue

        config = REPO_DIR / "configs" / "v3" / f"constrained_action_n{n_ris}.yaml"
        run_command(
            [
                sys.executable,
                "scripts/create_scenario_banks.py",
                "--config",
                config,
                "--output-dir",
                bank_dir,
                "--train-count",
                "10000",
                "--validation-count",
                "1000",
                "--test-count",
                "1000",
            ],
            cwd=REPO_DIR,
            log_path=log_dir / f"N{n_ris}.log",
        )

def write_stage_manifest(
    *,
    stage_root: Path,
    stage_id: str,
    commit: str,
    n_values: Sequence[int],
    extra: dict[str, object] | None = None,
) -> None:
    """Write machine-readable provenance consumed by notebook 06."""
    import torch

    payload: dict[str, object] = {
        "stage_id": stage_id,
        "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "repository": REPO_URL,
        "repository_ref": REPO_REF,
        "repository_commit": commit,
        "n_values": list(map(int, n_values)),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device": (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        ),
    }
    if extra:
        payload.update(extra)
    (stage_root / "STAGE_MANIFEST.json").write_text(
        json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8"
    )


In [ ]:
METHOD = "analytical_ris"
STAGE_ID = "analytical_ris"
N_VALUES = (16, 32, 64, 96, 128)
STAGE_ROOT = Path("/kaggle/working/thesis_analytical_ris")
STAGE_ROOT.mkdir(parents=True, exist_ok=True)

commit = clone_and_install()
create_locked_banks(N_VALUES, STAGE_ROOT)


In [ ]:
import matplotlib.pyplot as plt

MAX_PARALLEL_SOLVERS = 4

def chunk_plan(method: str, n_ris: int) -> list[tuple[int, int]]:
    """Return a resume-friendly scenario partition for one deterministic solver."""
    if method == "analytical_ris":
        return [(0, 1000)]
    # Four chunks exploit the Kaggle CPU cores without changing any scenario.
    return [(0, 250), (250, 250), (500, 250), (750, 250)]

def valid_solver_chunk(
    path: Path,
    *,
    method: str,
    start: int,
    count: int,
) -> bool:
    """Validate row count, exact scenario coverage, method label, and finiteness."""
    if not path.exists():
        return False
    try:
        frame = pd.read_csv(path)
        scenarios = sorted(
            pd.to_numeric(frame["scenario"], errors="raise").astype(int).tolist()
        )
        expected = list(range(start, start + count))
        metrics = frame[
            ["sum_rate", "qos_fraction", "all_qos", "violation", "solve_ms"]
        ].apply(pd.to_numeric, errors="coerce")
        return (
            len(frame) == count
            and scenarios == expected
            and set(frame["method"].astype(str).str.lower()) == {method}
            and np.isfinite(metrics.to_numpy(dtype=float)).all()
            and frame["bank_checksum"].astype(str).nunique() == 1
        )
    except Exception:
        return False

def run_solver_chunk(
    method: str,
    n_ris: int,
    start: int,
    count: int,
    stage_root: Path,
) -> Path:
    """Execute and audit one deterministic solver chunk."""
    config = REPO_DIR / "configs" / "v3" / f"constrained_action_n{n_ris}.yaml"
    bank = REPO_DIR / "artifacts" / "scenario_banks" / f"N{n_ris}_test.npz"
    output_dir = stage_root / "traditional_outputs" / method / f"N{n_ris}"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_csv = output_dir / f"part_{start}_{count}.csv"

    if valid_solver_chunk(
        output_csv, method=method, start=start, count=count
    ):
        print(f"Reuse {method} N={n_ris}, scenarios={start}:{start + count}")
        return output_csv

    run_command(
        [
            sys.executable,
            "scripts/run_solver.py",
            "--method",
            method,
            "--config",
            config,
            "--bank",
            bank,
            "--seed",
            "10000",
            "--start",
            str(start),
            "--count",
            str(count),
            "--output",
            output_csv,
        ],
        cwd=REPO_DIR,
        log_path=output_csv.with_suffix(".log"),
        extra_env={
            "OMP_NUM_THREADS": "1",
            "MKL_NUM_THREADS": "1",
            "OPENBLAS_NUM_THREADS": "1",
        },
    )
    if not valid_solver_chunk(
        output_csv, method=method, start=start, count=count
    ):
        raise RuntimeError(f"Invalid solver output: {output_csv}")
    return output_csv

def run_solver_stage(
    method: str,
    n_values: Sequence[int],
    stage_root: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Evaluate one baseline on all 5,000 locked test scenarios and summarize it."""
    jobs = [
        (int(n), int(start), int(count))
        for n in n_values
        for start, count in chunk_plan(method, int(n))
    ]
    outputs: list[Path] = []

    with concurrent.futures.ThreadPoolExecutor(
        max_workers=MAX_PARALLEL_SOLVERS
    ) as executor:
        future_map = {
            executor.submit(
                run_solver_chunk, method, n, start, count, stage_root
            ): (n, start, count)
            for n, start, count in jobs
        }
        for future in concurrent.futures.as_completed(future_map):
            outputs.append(future.result())

    frames = [pd.read_csv(path) for path in sorted(outputs)]
    raw = pd.concat(frames, ignore_index=True)
    raw["n_ris"] = pd.to_numeric(raw["n_ris"], errors="raise").astype(int)
    raw["scenario"] = pd.to_numeric(raw["scenario"], errors="raise").astype(int)
    raw = raw.sort_values(["n_ris", "scenario"]).reset_index(drop=True)

    for n_ris in n_values:
        group = raw[raw["n_ris"] == n_ris]
        scenarios = sorted(group["scenario"].tolist())
        if len(group) != 1000 or scenarios != list(range(1000)):
            raise RuntimeError(f"Incomplete {method} coverage for N={n_ris}")

    raw_path = stage_root / f"{method.upper()}_RAW_ALL.csv"
    raw.to_csv(raw_path, index=False)

    rows: list[dict[str, object]] = []
    for n_ris, group in raw.groupby("n_ris", sort=True):
        row: dict[str, object] = {
            "method": method,
            "n_ris": int(n_ris),
            "scenarios": int(len(group)),
            "bank_checksum": str(group["bank_checksum"].iloc[0]),
        }
        for metric in ["sum_rate", "qos_fraction", "all_qos", "violation", "solve_ms"]:
            values = pd.to_numeric(group[metric], errors="raise").to_numpy(float)
            row[f"{metric}_mean"] = float(np.mean(values))
            row[f"{metric}_std"] = float(np.std(values, ddof=1))
            row[f"{metric}_median"] = float(np.median(values))
            row[f"{metric}_p95"] = float(np.quantile(values, 0.95))
        rows.append(row)

    summary = pd.DataFrame(rows)
    summary.to_csv(stage_root / f"{method.upper()}_SUMMARY.csv", index=False)
    return raw, summary

def save_solver_figures(
    method: str,
    summary: pd.DataFrame,
    stage_root: Path,
) -> None:
    """Export paper-ready quality and solve-time figures for one baseline."""
    figure_dir = stage_root / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)

    for metric, ylabel, use_log in [
        ("sum_rate_mean", "Mean sum-rate", False),
        ("qos_fraction_mean", "Mean QoS fraction", False),
        ("violation_mean", "Mean QoS violation", True),
        ("solve_ms_mean", "Mean solve time (ms)", True),
    ]:
        fig, ax = plt.subplots(figsize=(6.4, 4.2))
        ax.plot(summary["n_ris"], summary[metric], marker="o")
        if use_log:
            ax.set_yscale("log")
        ax.set_xlabel("Number of STAR-RIS elements, N")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.25)
        fig.tight_layout()
        fig.savefig(
            figure_dir / f"{method}_{metric}.png",
            dpi=300,
            bbox_inches="tight",
        )
        fig.savefig(
            figure_dir / f"{method}_{metric}.pdf",
            bbox_inches="tight",
        )
        plt.close(fig)


## Run, audit, summarize, and plot

In [ ]:
raw, summary = run_solver_stage(METHOD, N_VALUES, STAGE_ROOT)
save_solver_figures(METHOD, summary, STAGE_ROOT)
display(summary)


In [ ]:
write_stage_manifest(
    stage_root=STAGE_ROOT,
    stage_id=STAGE_ID,
    commit=commit,
    n_values=N_VALUES,
    extra={
        "method": METHOD,
        "seed": 10000,
        "expected_test_scenarios_per_n": 1000,
        "max_parallel_solvers": MAX_PARALLEL_SOLVERS,
    },
)
print("Stage output:", STAGE_ROOT)
